# Check a PDF with statcheck-ml

This notebook takes a PDF of a paper and returns every statistical result in
it, with a verdict for each one.

**The package has two halves, and the difference matters.**

| Half | How it works |
|---|---|
| **Finding** a result | learned, because a pattern cannot read an operator the PDF conversion destroyed |
| **Checking** a result | closed-form mathematics, never learned |

No model output reaches a verdict. The model says *where* a result is. The
arithmetic says whether it is right.

## Install

From the repository root:

```
pip install -e statcheck-ml
```

`pymupdf` and `torch` are needed to read a PDF and to run the model.

In [1]:
import statcheck_ml

print("statcheck-ml", statcheck_ml.__version__)

statcheck-ml 0.1.0


## 1. Start without a PDF

`Pipeline` also accepts plain text. With no model it uses the pattern alone,
which is what the original `statcheck` does. That makes a good starting point,
because it shows what the pattern can and cannot read.

In [2]:
from statcheck_ml import Pipeline

passage = (
    "Reaction times differed between the groups, t(23) = 2.45, p = .022. "
    "The effect of condition was reliable, F(2, 30) = 5.10, p = .012. "
    "Accuracy did not differ, chi2(1, N = 223) = 8.69, p = .003."
)

pattern_only = Pipeline()          # no model_path, so the pattern works alone
report = pattern_only.run_text(passage)

for r in report["results"]:
    print(f"{r['test_type']:>5}({r['df1']}, {r['df2']}) = {r['statistic']}, "
          f"p {r['p_operator']} {r['p_value']}  ->  {r['verdict']}")

    t(23.0, None) = 2.45, p = 0.022  ->  consistent
    f(2.0, 30.0) = 5.1, p = 0.012  ->  consistent
 chi2(1.0, None) = 8.69, p = 0.003  ->  consistent


## 2. The problem the model solves

A publisher sets an operator in a font that carries no ToUnicode map. The
character then has **no Unicode value at all**, and the converter writes
whatever is left. PyMuPDF writes a control character.

The information is absent from the file. No converter recovers it by reading
harder. More than half of the results in a real corpus are damaged this way.

Below, `\x03` and `\x04` stand where `=` and `<` belong.

In [3]:
damaged = (
    "Performance improved, F(1, 17) \x03 35.72, p \x04 .0005. "
    "The groups differed, t(23) \x03 2.45, p \x03 .022. "
    "Recall was higher, F(2, 30) \x03 5.10, p \x03 .012."
)

print("raw text, as the PDF gives it:")
print(" ", repr(damaged[:60]), "...")
print()

# `Pipeline` repairs the text by default. Turn that off, to show what the
# pattern alone can read.
raw_pattern = Pipeline(repair_text=False)
print("pattern alone, no repair :", len(raw_pattern.run_text(damaged)["results"]),
      "results")
print("pattern with the repair  :", len(pattern_only.run_text(damaged)["results"]),
      "results")

raw text, as the PDF gives it:
  'Performance improved, F(1, 17) \x03 35.72, p \x04 .0005. The group' ...

pattern alone, no repair : 0 results
pattern with the repair  : 3 results


### The repair stage recovers the operator

The numbers survive even when the operator does not. The repair stage tries
every candidate operator, recomputes the p-value for each, and keeps the
mapping the arithmetic supports best.

**This is not machine learning.** It is arithmetic, so no annotator and no
model takes part in it.

In [4]:
repaired_text, info = pattern_only.repair(damaged)

print("after repair:")
print(" ", repr(repaired_text[:60]), "...")
print()
print("the mapping the arithmetic chose:", info.get("chosen"))
print("results it could test           :", info.get("testable"))
print("share that agreed               :", round(info.get("agreement", 0), 3))

after repair:
  'Performance improved, F(1, 17) = 35.72, p < .0005. The group' ...

the mapping the arithmetic chose: {"'\\x03'": '=', "'\\x04'": '<'}
results it could test           : 3
share that agreed               : 1.0


## 3. Load the model

`final-crf-aug` is the model for every port, including the browser. It carries
a CRF, so pass `use_crf=True`.

| Model | Parameters | ONNX | Holdout F2 |
|---|---|---|---|
| **`final-crf-aug`** | 616,072 | 2.47 MB | **0.931** |
| `final-gru-crf` | 466,696 | 1.87 MB | 0.918 |

The smaller model saves 600 kB and 0.6 seconds over a whole document, measured,
and costs 0.013 of F2. Use it only where 600 kB genuinely matters.

In [5]:
from pathlib import Path

MODEL = Path("../models/final-crf-aug/model.pt")

pipeline = Pipeline(model_path=str(MODEL), use_crf=True)
print("loaded", MODEL.name, "with a CRF")

loaded model.pt with a CRF


## 4. Read a PDF

Point `PDF_PATH` at any paper. The pipeline runs six stages:

| Stage | What it does |
|---|---|
| extract | a named PDF engine turns the PDF into text |
| normalize | the text is made engine independent |
| repair | damaged operators are restored by arithmetic |
| prefilter | about 1 line in 700 holds a result; the rest is dropped |
| find | the pattern reads what it can, the model reads the rest |
| check | the p-value is recomputed |

In [6]:
from statcheck_ml import summarise

# A small paper written for this notebook. Put your own PDF here instead.
PDF_PATH = "sample_paper.pdf"

if not Path(PDF_PATH).exists():
    import make_sample_paper
    make_sample_paper.main()

report = pipeline.run_pdf(PDF_PATH)
print(summarise(report))

sample_paper.pdf
  characters        : 2,019
  windows kept      : 13 of 41 lines
  found by pattern  : 7
  found by model    : 3
  verdicts          : consistent=6, undecidable=3, decision_error=1
  parts not found   : df1=2, p_value=2
  seconds           : 1.11


## 5. The results

Every result carries where it came from and how it was found, so a reader can
go back to the page and judge for themselves.

In [7]:
results = report["results"]
print(f"{len(results)} results\n")

head = f"{'test':>6} {'statistic':>10} {'df1':>5} {'df2':>6} {'p':>10} " \
       f"{'source':>8}  verdict"
print(head)
print("-" * len(head))
for r in results[:20]:
    df2 = "" if r["df2"] is None else f"{r['df2']:.0f}"
    df1 = "" if r["df1"] is None else f"{r['df1']:.0f}"
    p = "" if r["p_value"] is None else f"{r['p_operator'] or ''}{r['p_value']}"
    print(f"{r['test_type']:>6} {r['statistic']:>10} {df1:>5} {df2:>6} "
          f"{p:>10} {r['source']:>8}  {r['verdict']}")

10 results

  test  statistic   df1    df2          p   source  verdict
-----------------------------------------------------------
     t       2.45    23            =0.022  pattern  consistent
     f        5.1     2     30     =0.012  pattern  consistent
  chi2       8.69     1            =0.003  pattern  consistent
     r       0.42                   =0.02    model  undecidable
     t        1.8    46             =0.04  pattern  decision_error
     f        9.2     1    118     =0.003  pattern  consistent
     z       1.96                   =0.05  pattern  consistent
     t       4.15    19            <0.001  pattern  consistent
     f       2.71     3     92               model  undecidable
     t        3.6                            model  undecidable


## 6. What each stage did

Every stage records its own work, so a surprising answer can be traced to the
stage that caused it.

In [8]:
import json

for name, info in report["stages"].items():
    print(f"{name}:")
    print(" ", json.dumps(info, default=str)[:200])

normalize:
  {"lines_before": 41, "lines_after": 41, "operators_renamed": {}}
repair:
  {"reason": "nothing to infer", "results": 6, "fallback": "simple rule", "chosen": {}, "replacements": 0}
prefilter:
  {"lines": 41, "windows_kept": 13}
find:
  {"by_pattern": 7, "by_model": 3}
check:
  {"consistent": 6, "undecidable": 3, "decision_error": 1}
not_found:
  {"df1": 2, "p_value": 2}


## 7. The verdicts

| Verdict | Meaning |
|---|---|
| `consistent` | the reported p-value agrees with the recomputed one |
| `inconsistent` | they disagree |
| `decision_error` | they disagree **about significance**, which is the serious case |
| `undecidable` | a part needed for the recomputation was not found |

A result whose degrees of freedom were not found is real but cannot be checked.
Report the two counts apart, and **never treat `undecidable` as `consistent`.**

Every result carries a `missing` field naming the parts that were not found,
and the run report counts them under `not_found`.

**The tool says what it observed, never why.** `no p-value found beside this
result` is the observation. Whether the author omitted the number, the font
destroyed it, or the extraction missed it is a different question, and the
`quote` beside each result is what answers it.

In [9]:
from collections import Counter

counts = Counter(r["verdict"] for r in results)
for verdict, n in counts.most_common():
    print(f"{verdict:>15}: {n}")

problems = [r for r in results
            if r["verdict"] in ("inconsistent", "decision_error")]
print(f"\n{len(problems)} result(s) worth a human look:\n")
for r in problems[:5]:
    print(f"  reported: {r['quote'][:70]!r}")
    print(f"  recomputed p = {r['computed_p']}")
    print(f"  {r['reason']}\n")

     consistent: 6
    undecidable: 3
 decision_error: 1

1 result(s) worth a human look:

  reported: 't(46) = 1.80, p = .04'
  recomputed p = 0.07842066481562293
  the reported and computed p-values disagree about significance



## 8. The PDF engine is a choice, not a detail

Each engine returns different text, and no engine serves all three ports.
Python defaults to PyMuPDF because the models were trained on its output.

| Engine | Recall | Note |
|---|---|---|
| `pymupdf` | 0.929 | the default; AGPL-3.0 |
| `pdfium` | 0.916 | permissive licence, needs `pypdfium2` |
| `poppler` | 0.885 | what R gets; needs `pdftotext` on the path |

In [10]:
for engine in ("pymupdf", "pdfium"):
    try:
        other = Pipeline(model_path=str(MODEL), use_crf=True, engine=engine)
        out = other.run_pdf(PDF_PATH)
        print(f"{engine:>8}: {len(out['results']):3d} results, "
              f"{out['characters']:,} characters")
    except ImportError as exc:
        print(f"{engine:>8}: not installed ({exc})")

 pymupdf:  10 results, 2,019 characters


  pdfium:   9 results, 1,979 characters


## 9. A folder of PDFs

The pipeline holds no state between documents, so a loop is enough.

In [11]:
import csv
from pathlib import Path

def check_folder(folder, out_csv="results.csv", limit=None):
    """Check every PDF in a folder and write one row for each result."""
    rows = []
    pdfs = sorted(Path(folder).glob("*.pdf"))[:limit]
    for pdf in pdfs:
        try:
            out = pipeline.run_pdf(str(pdf))
        except Exception as exc:      # one bad file must not stop the run
            print(f"  skipped {pdf.name}: {exc}")
            continue
        for r in out["results"]:
            rows.append({"document": pdf.name, **r})
        print(f"  {pdf.name}: {len(out['results'])} results")

    if rows:
        with open(out_csv, "w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        print(f"\nwrote {len(rows)} rows to {out_csv}")
    return rows

# check_folder("path/to/pdfs", limit=3)
print("call check_folder() with a folder of PDFs")

call check_folder() with a folder of PDFs


## 9b. Why is everything the model found `undecidable`?

Look again at the table in section 5. Every result the **pattern** found is
checked, and every result the **model** found is `undecidable`. That looks like
the model is the weak half. It is not, and the reason is the order of the
stages.

**The pattern runs first, and it runs on repaired text.** Stages 2 and 3 clean
the operators before the pattern sees them, so the pattern reads every
well-formed result and the model receives only what is left over. In a clean
paper what is left over is exactly the awkward material: a correlation with no
degrees of freedom, a result split by a line break, a number from a table.

Turn the stages off one at a time and the halves separate. The paper below
carries the same results with their operators destroyed.

In [12]:
DAMAGED_PDF = "sample_paper_damaged.pdf"

if not Path(DAMAGED_PDF).exists():
    import make_sample_paper
    make_sample_paper.make_damaged()

print("the text the engine returns:")
print(" ", repr(Pipeline.extract_text(DAMAGED_PDF)[:66]))
print()

cases = [
    ("pattern only, repair ON ", Pipeline()),
    ("pattern only, repair OFF", Pipeline(repair_text=False)),
    ("model only,   repair OFF", Pipeline(model_path=str(MODEL), use_crf=True,
                                          repair_text=False, use_pattern=False)),
    ("full cascade, repair ON ", Pipeline(model_path=str(MODEL), use_crf=True)),
]
print(f"{'configuration':<26}{'found':>7}{'checkable':>11}")
print("-" * 44)
for name, engine in cases:
    got = engine.run_pdf(DAMAGED_PDF)["results"]
    checkable = sum(1 for r in got if r["verdict"] != "undecidable")
    print(f"{name:<26}{len(got):>7}{checkable:>11}")

the text the engine returns:
  'A paper whose operators the conversion destroyed\nReaction times di'

configuration               found  checkable
--------------------------------------------
pattern only, repair ON         5          5
pattern only, repair OFF        0          0


model only,   repair OFF        5          0


full cascade, repair ON         5          5


### What the four rows mean

| Configuration | Found | Checkable | What it shows |
|---|---|---|---|
| pattern only, repair ON | 5 | 5 | the repair, not the pattern, is doing the reading |
| pattern only, repair OFF | **0** | 0 | **the pattern cannot read damaged text at all** |
| model only, repair OFF | **5** | 0 | **the model reads the damage directly** |
| full cascade | 5 | 5 | both halves, which is what ships |

**The model finds. The repair makes the finding checkable.** Neither alone is
enough: the model reads the result but cannot recompute a p-value while the
operator is still destroyed, and the repair cannot fix an operator inside a
result that nobody found.

On the holdout this is the whole difference. The pattern alone finds 2 of 164
damaged results. The cascade finds 152.

## 10. What the tool gets wrong on this very document

A tutorial that only shows success teaches nothing. This paper was written so
that three known faults appear in the output. All three are `undecidable`,
which is why that verdict must never be read as `consistent`.

| Result | What happened |
|---|---|
| `r = .42` | A correlation printed with no degrees of freedom. **It is found and it cannot be checked**, because the p-value cannot be recomputed without them. |
| `F(3, 92) = 2.71` | The statistic is on one line and `p = .049` on the next. The window ended between them, so the p-value never reached the model. |
| `t = 3.6` | A false positive. `3.6` is a standard deviation in Table 1, and the model read it as a statistic. |

The three look the same in the output, and they are not the same thing:

- The first is the paper. Nothing can recover a number that is not printed.
- The second is this tool. The window cut the result in half.
- The third is this tool. It read a table cell as a statistic.

**The output does not claim to know which.** It reports `no p-value found` and
gives the quote. A reader decides.

On 60 real papers the window fault of the second row **did not occur once**, so
it is a demonstrated limit rather than a common one. The false positive of the
third row is the cost of tuning for recall: a screening tool should rather show
a reader one extra number than hide a real error.

In [13]:
for r in results:
    if r["verdict"] != "undecidable":
        continue
    df = "no df" if r["df1"] is None else "df=%.0f" % r["df1"]
    p = "no p-value" if r["p_value"] is None else "p=%s" % r["p_value"]
    print("%5s = %-7s %-8s %-12s line %s"
          % (r["test_type"], r["statistic"], df, p, r["line"]))
    print("      quote: %r" % r["quote"][:60])
    print()

    r = 0.42    no df    p=0.02       line 17
      quote: 'r = .42, p = .02'

    f = 2.71    df=3     no p-value   line 22
      quote: 'F(3, 92) = 2.71'

    t = 3.6     no df    no p-value   line 26
      quote: '3.6'



## What to remember

1. **The verdict is arithmetic.** The model finds a result. It never decides
   whether the result is right.
2. **The prefilter caps recall.** Text it discards can never be recovered, so
   it is tuned for recall alone.
3. **`undecidable` is not `consistent`.** A result without degrees of freedom
   cannot be checked at all.
4. **Every label behind the measurements is a machine label.** The measured
   ceiling is 90.5% on the holdout. Read every score against it.
5. **A flagged result is a result to read, not a verdict on the author.** The
   tool reports a disagreement between two numbers, and nothing more.

Measured against the R package on the same holdout: recall 0.937 against 0.187,
and on damaged text 0.926 against 0.012.

`REPORT.md` holds the method and every number behind those figures.